# 01. PostgreSQL 기초
> Day 1 · 2H · 소요 약 50분

## 학습 목표

- Neon PostgreSQL 인스턴스를 생성하고 Colab에서 접속할 수 있다.
- `psycopg2`와 `SQLAlchemy`의 차이를 이해하고 상황에 맞게 사용한다.
- `SELECT` / `WHERE` / `ORDER BY` / `LIMIT` / `EXPLAIN` 기본을 실행할 수 있다.
- 병원 샘플 DB를 자신의 Neon 인스턴스에 적재한다.

> **중요:** 이 노트북은 Day 1·2 강의(노트북 02~09)의 모든 후속 실습에 사용할 **병원 데이터베이스**를 여러분의 Neon 인스턴스에 적재합니다. 다른 노트북을 열기 전에 **이 노트북을 먼저 실행하세요**.

In [ ]:
%pip install -q psycopg2-binary sqlalchemy pandas tabulate

In [ ]:
# 이 셀은 "API 키 / DB 접속 정보 같은 비밀(Secret) 을 안전하게 읽어 오는" 도우미입니다.
# Colab 의 [🔑 Secrets] 탭에 넣어 둔 값이 있으면 그걸 읽고, 없으면 입력창을 띄워 받습니다.
# 코드에 키를 직접 적어 두면 노트북을 공유할 때 노출될 위험이 있어 매 노트북 첫 셀에서 이렇게 처리합니다.
import os  # os.environ 으로 운영체제 환경변수에 접근하는 표준 모듈


def _load_secret(key: str, required: bool = True) -> None:
    """환경변수 `key` 를 채워 넣는다. Colab Secrets → getpass 입력 순으로 시도."""
    # 1) 이미 환경변수에 있으면 그대로 둔다 (셀을 다시 돌릴 때 매번 묻는 일을 막음).
    if os.environ.get(key):
        return
    value = None
    # 2) Colab 환경이면 좌측 [🔑 Secrets] 패널에서 읽어 본다.
    try:
        from google.colab import userdata  # type: ignore  (Colab 전용 모듈)
        value = userdata.get(key)
    except Exception:
        # 로컬 PC 등 Colab 이 아닌 환경에서는 import 자체가 실패 → 무시
        value = None
    # 3) 그래도 못 찾으면 비밀번호처럼 입력값이 화면에 안 보이는 입력창을 띄운다.
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    # 4) 값이 있으면 환경변수에 저장, 없는데 필수면 명시적으로 에러를 던진다.
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")


# 이 노트북은 Neon DSN 만 필수. OpenAI 키는 다음 노트북에서 본격적으로 쓰이므로 선택.
_load_secret("NEON_DSN", required=True)
_load_secret("OPENAI_API_KEY", required=False)

print("Environment ready.")

## 왜 PostgreSQL + Neon인가?

| 비교 항목 | PostgreSQL | MySQL | SQLite |
|---|---|---|---|
| 라이선스 | 완전 무료 (BSD) | 이중 라이선스 | 퍼블릭 도메인 |
| 표준 SQL 준수 | 매우 높음 | 보통 | 제한적 |
| 윈도우 함수 | 완전 지원 | 8.0+ 지원 | 제한적 |
| JSON 지원 | JSONB | JSON | JSON |
| CTE (`WITH`) | 재귀 포함 | 8.0+ | 지원 |
| `COMMENT ON` | 지원 | 미지원 | 미지원 |

PostgreSQL은 `COMMENT ON` 지원 덕분에 **AI 가독성 스키마** 구축에 이상적입니다.

**Neon**은 브라우저에서 30초면 무료 인스턴스를 만들 수 있고, Colab·로컬 어디서나 같은 DSN으로 접속할 수 있어 학습용으로 최적입니다.

In [ ]:
# SQLAlchemy 로 Neon PostgreSQL 에 접속해 본다.
#   create_engine: DB 접속 객체(엔진) 를 만든다. 실제 연결은 .connect() 호출 시점에 발생한다.
#   text         : 문자열 SQL 을 SQLAlchemy 가 안전하게 다룰 수 있도록 감싸는 함수.
from sqlalchemy import create_engine, text
import pandas as pd  # 결과를 표(DataFrame) 형태로 다루기 위해 미리 임포트

# os.environ["NEON_DSN"] 은 위 셀에서 환경변수에 적재한 Neon Connection String.
# DSN(Data Source Name) 은 "어디로/누가/어떤 DB로" 한 줄에 담은 표준 형식입니다.
engine = create_engine(os.environ["NEON_DSN"])

# `with engine.connect() as conn:` 패턴
#   - 블록을 빠져나갈 때 커넥션을 자동으로 풀어 준다 (자원 누수 방지).
#   - 여기서는 단순히 SELECT 두 번을 날려 DB 가 살아 있고 시간/버전을 정상 응답하는지 확인.
with engine.connect() as conn:
    # .execute() 는 결과를 ResultProxy 로 돌려준다. .fetchone() 은 첫 행만 받기.
    # [0] 인덱싱은 행에서 첫 컬럼만 추출 — 우리는 단일 컬럼만 조회하므로 충분.
    version = conn.execute(text("SELECT version()")).fetchone()[0]
    now = conn.execute(text("SELECT NOW()")).fetchone()[0]
    print(f"PostgreSQL version: {version}")
    print(f"Server time: {now}")

In [ ]:
# psycopg2 는 PostgreSQL 전용 저수준 드라이버입니다.
# SQLAlchemy 가 내부적으로 이 라이브러리를 사용해 실제 DB 와 통신하지만,
# 직접 써 보면 "원시 형태"가 어떻게 생겼는지 감이 옵니다.
import psycopg2

# 직접 커넥션 생성 → 커서 객체 → SQL 실행 → 결과 fetch → 정리(close).
# 이 4단계가 모든 DB API 의 공통 패턴입니다 (psycopg2, mysql-connector, sqlite3 등).
conn_pg = psycopg2.connect(os.environ["NEON_DSN"])
cur = conn_pg.cursor()                                 # SQL 을 실행할 커서
cur.execute("SELECT current_database(), current_user")  # 두 함수의 결과를 한 행으로 반환
db_name, user_name = cur.fetchone()                    # 튜플 언패킹: (db, user)
print(f"database={db_name}, user={user_name}")
cur.close()        # 커서 정리
conn_pg.close()    # 커넥션 정리 — 깜빡하면 커넥션 풀이 점점 차오릅니다

### SQLAlchemy vs psycopg2 — 언제 무엇을 쓰나?

- **`psycopg2`** — PostgreSQL용 저수준 드라이버. 커넥션/커서를 직접 제어.
- **`SQLAlchemy`** — 여러 DB를 추상화한 고수준 래퍼. ORM과 편의 기능 제공.

본 강의에서는 SQLAlchemy를 기본으로 사용하고, psycopg2는 드라이버로만 뒤에서 동작한다고 이해하면 됩니다.

## 병원 DB 스키마 개요 (텍스트 ERD)

```
departments  ─┬─< doctors ─< visits >─ patients
              │                │
              │                └── diagnoses
              │
              (진료과별 의사/의사별 진료 기록/진료별 진단)
```

총 5개 테이블: `departments`, `doctors`, `patients`, `visits`, `diagnoses`

**다음 셀에서 DDL을 실행하여 스키마를 만듭니다.** `DROP TABLE IF EXISTS`로 시작하므로 여러 번 재실행해도 안전합니다.

In [ ]:
# 병원 DB 의 DDL(Data Definition Language) — 테이블의 "구조"를 정의하는 SQL 입니다.
# 이 셀은 한 번 실행해서 테이블 5 개와 그 위의 모든 COMMENT 를 한꺼번에 만듭니다.
#
# 이 DDL 의 학습 포인트 4가지:
#   1) `DROP TABLE IF EXISTS … CASCADE` — 이미 있어도 에러 없이 지우고, 자식 테이블의
#      외래키도 같이 정리해서 셀을 여러 번 돌려도 안전합니다(idempotent).
#   2) `SERIAL PRIMARY KEY` — 자동 증가 정수 + 기본키. PostgreSQL 만의 편의 문법.
#   3) `REFERENCES`(외래키) — 다른 테이블의 PK 와 연결. AI 가 JOIN 경로를 추론하는 핵심 단서.
#   4) `CHECK (… IN (…))` — 컬럼 값에 미리 정한 도메인을 강제(예: gender 는 'M' 또는 'F').
#   5) `COMMENT ON COLUMN` — 컬럼에 설명을 붙임. LLM 이 "gender 가 뭐지?" 추측하지 않게 한다.
hospital_ddl = """
-- Drop in reverse FK order
DROP TABLE IF EXISTS diagnoses CASCADE;
DROP TABLE IF EXISTS visits CASCADE;
DROP TABLE IF EXISTS doctors CASCADE;
DROP TABLE IF EXISTS patients CASCADE;
DROP TABLE IF EXISTS departments CASCADE;

CREATE TABLE departments (
    department_id   SERIAL PRIMARY KEY,
    name            VARCHAR(50) NOT NULL,
    floor           INT,
    phone           VARCHAR(20)
);
COMMENT ON TABLE  departments            IS '병원의 진료과 정보';
COMMENT ON COLUMN departments.department_id IS '진료과 고유 식별자';
COMMENT ON COLUMN departments.name         IS '진료과명 (예: 내과, 외과, 소아과)';
COMMENT ON COLUMN departments.floor        IS '진료과 위치 층수';
COMMENT ON COLUMN departments.phone        IS '진료과 대표 전화번호';

CREATE TABLE doctors (
    doctor_id       SERIAL PRIMARY KEY,
    name            VARCHAR(100) NOT NULL,
    department_id   INT NOT NULL REFERENCES departments(department_id),
    specialty       VARCHAR(100),
    hire_date       DATE NOT NULL,
    salary          NUMERIC(12,2)
);
COMMENT ON TABLE  doctors               IS '의사 정보';
COMMENT ON COLUMN doctors.doctor_id     IS '의사 고유 식별자';
COMMENT ON COLUMN doctors.name          IS '의사 이름';
COMMENT ON COLUMN doctors.department_id IS '소속 진료과 (departments 참조)';
COMMENT ON COLUMN doctors.specialty     IS '세부 전공 (예: 심장내과)';
COMMENT ON COLUMN doctors.hire_date     IS '입사일';
COMMENT ON COLUMN doctors.salary        IS '월급 (원)';

CREATE TABLE patients (
    patient_id      SERIAL PRIMARY KEY,
    name            VARCHAR(100) NOT NULL,
    birth_date      DATE NOT NULL,
    gender          CHAR(1) NOT NULL CHECK (gender IN ('M','F')),
    phone           VARCHAR(20),
    address         VARCHAR(200),
    blood_type      VARCHAR(3) CHECK (blood_type IN ('A','B','O','AB')),
    created_at      TIMESTAMP DEFAULT NOW()
);
COMMENT ON TABLE  patients             IS '환자 기본 정보';
COMMENT ON COLUMN patients.patient_id  IS '환자 고유 식별자';
COMMENT ON COLUMN patients.name        IS '환자 이름';
COMMENT ON COLUMN patients.birth_date  IS '생년월일';
COMMENT ON COLUMN patients.gender      IS '성별: M=남성, F=여성';
COMMENT ON COLUMN patients.phone       IS '연락처';
COMMENT ON COLUMN patients.address     IS '주소';
COMMENT ON COLUMN patients.blood_type  IS '혈액형: A, B, O, AB';
COMMENT ON COLUMN patients.created_at  IS '환자 등록 일시';

CREATE TABLE visits (
    visit_id        SERIAL PRIMARY KEY,
    patient_id      INT NOT NULL REFERENCES patients(patient_id),
    doctor_id       INT NOT NULL REFERENCES doctors(doctor_id),
    visit_date      DATE NOT NULL,
    visit_type      VARCHAR(20) NOT NULL CHECK (visit_type IN ('outpatient','inpatient','emergency')),
    status          VARCHAR(20) NOT NULL DEFAULT 'scheduled'
                    CHECK (status IN ('scheduled','completed','cancelled','no_show')),
    chief_complaint TEXT,
    cost            NUMERIC(10,2)
);
COMMENT ON TABLE  visits                   IS '환자 진료 방문 기록';
COMMENT ON COLUMN visits.visit_id          IS '방문 고유 식별자';
COMMENT ON COLUMN visits.patient_id        IS '방문 환자 (patients 참조)';
COMMENT ON COLUMN visits.doctor_id         IS '담당 의사 (doctors 참조)';
COMMENT ON COLUMN visits.visit_date        IS '진료 날짜';
COMMENT ON COLUMN visits.visit_type        IS '진료 유형: outpatient=외래, inpatient=입원, emergency=응급';
COMMENT ON COLUMN visits.status            IS '진료 상태: scheduled=예약, completed=완료, cancelled=취소, no_show=미방문';
COMMENT ON COLUMN visits.chief_complaint   IS '주요 증상/호소 내용';
COMMENT ON COLUMN visits.cost              IS '진료비 (원)';

CREATE TABLE diagnoses (
    diagnosis_id    SERIAL PRIMARY KEY,
    visit_id        INT NOT NULL REFERENCES visits(visit_id),
    icd_code        VARCHAR(10) NOT NULL,
    description     VARCHAR(200) NOT NULL,
    severity        VARCHAR(10) CHECK (severity IN ('mild','moderate','severe'))
);
COMMENT ON TABLE  diagnoses               IS '진료 시 내려진 진단 기록';
COMMENT ON COLUMN diagnoses.diagnosis_id  IS '진단 고유 식별자';
COMMENT ON COLUMN diagnoses.visit_id      IS '관련 방문 (visits 참조)';
COMMENT ON COLUMN diagnoses.icd_code      IS 'ICD-10 질병 분류 코드';
COMMENT ON COLUMN diagnoses.description   IS '진단명 (한국어)';
COMMENT ON COLUMN diagnoses.severity      IS '중증도: mild=경증, moderate=중등, severe=중증';
"""

# `engine.begin()` 은 트랜잭션 블록을 연다 — 블록을 정상 종료하면 자동 COMMIT,
# 도중에 예외가 나면 자동 ROLLBACK. DDL 같은 "한 번에 모두 적용 또는 모두 취소" 작업에 적합.
with engine.begin() as conn:
    conn.execute(text(hospital_ddl))
print("Hospital DDL applied.")

In [ ]:
seed_data = """
-- Departments
INSERT INTO departments (name, floor, phone) VALUES
('내과', 3, '02-1234-1001'),
('외과', 4, '02-1234-1002'),
('소아과', 2, '02-1234-1003'),
('정형외과', 4, '02-1234-1004'),
('피부과', 2, '02-1234-1005'),
('신경과', 5, '02-1234-1006'),
('산부인과', 3, '02-1234-1007'),
('안과', 2, '02-1234-1008');

-- Doctors
INSERT INTO doctors (name, department_id, specialty, hire_date, salary) VALUES
('김철수', 1, '심장내과', '2015-03-01', 8500000),
('이영희', 1, '호흡기내과', '2018-07-15', 7200000),
('박민수', 2, '일반외과', '2012-01-10', 9000000),
('정수진', 2, '흉부외과', '2019-06-01', 7800000),
('최동현', 3, '소아청소년과', '2016-09-20', 7500000),
('강미래', 3, '신생아과', '2020-03-01', 6800000),
('윤성호', 4, '척추외과', '2014-05-15', 8800000),
('한지은', 4, '관절외과', '2017-11-01', 7600000),
('서준혁', 5, '일반피부과', '2021-01-15', 6500000),
('임하늘', 5, '미용피부과', '2022-03-01', 6200000),
('조태영', 6, '뇌신경과', '2013-08-20', 9200000),
('배수현', 6, '말초신경과', '2019-12-01', 7100000),
('노진우', 7, '산과', '2015-06-15', 8000000),
('유다정', 7, '부인과', '2018-09-01', 7400000),
('장세림', 8, '망막', '2016-04-10', 7900000),
('오현우', 8, '녹내장', '2020-07-01', 6900000),
('신민아', 1, '소화기내과', '2017-02-15', 7800000),
('권혁준', 2, '혈관외과', '2021-05-01', 6600000),
('문서영', 3, '소아알레르기', '2023-01-10', 6000000),
('황태윤', 6, '두통클리닉', '2022-06-15', 6300000);

-- Patients
INSERT INTO patients (name, birth_date, gender, phone, address, blood_type) VALUES
('홍길동', '1985-05-15', 'M', '010-1111-0001', '서울시 강남구', 'A'),
('김미영', '1990-08-22', 'F', '010-1111-0002', '서울시 서초구', 'B'),
('이준석', '1978-12-03', 'M', '010-1111-0003', '서울시 송파구', 'O'),
('박서연', '1995-03-17', 'F', '010-1111-0004', '서울시 마포구', 'AB'),
('정태호', '1982-07-30', 'M', '010-1111-0005', '경기도 성남시', 'A'),
('최유진', '2000-01-25', 'F', '010-1111-0006', '서울시 강동구', 'B'),
('강현우', '1975-11-08', 'M', '010-1111-0007', '서울시 중구', 'O'),
('윤서현', '1998-04-12', 'F', '010-1111-0008', '경기도 고양시', 'A'),
('임도윤', '1988-09-05', 'M', '010-1111-0009', '서울시 노원구', 'AB'),
('한소희', '1992-06-18', 'F', '010-1111-0010', '서울시 양천구', 'B'),
('조민기', '1970-02-28', 'M', '010-1111-0011', '서울시 용산구', 'A'),
('배지현', '2003-10-07', 'F', '010-1111-0012', '경기도 수원시', 'O'),
('서영준', '1980-08-14', 'M', '010-1111-0013', '서울시 동작구', 'B'),
('노혜린', '1996-12-25', 'F', '010-1111-0014', '서울시 관악구', 'A'),
('유재석', '1972-08-14', 'M', '010-1111-0015', '경기도 용인시', 'O'),
('장미란', '1983-11-09', 'F', '010-1111-0016', '서울시 영등포구', 'AB'),
('오승환', '1991-07-22', 'M', '010-1111-0017', '서울시 구로구', 'A'),
('신세경', '1999-02-03', 'F', '010-1111-0018', '경기도 안양시', 'B'),
('권상우', '1976-06-05', 'M', '010-1111-0019', '서울시 종로구', 'O'),
('문채원', '1987-11-13', 'F', '010-1111-0020', '서울시 성동구', 'A'),
('황정민', '1969-09-01', 'M', '010-1111-0021', '경기도 부천시', 'AB'),
('이나영', '1993-04-17', 'F', '010-1111-0022', '서울시 은평구', 'B'),
('김수현', '2001-12-08', 'M', '010-1111-0023', '서울시 광진구', 'O'),
('전지현', '1981-10-30', 'F', '010-1111-0024', '경기도 파주시', 'A'),
('송중기', '1986-09-19', 'M', '010-1111-0025', '서울시 강서구', 'B'),
('한가인', '1994-07-06', 'F', '010-1111-0026', '서울시 도봉구', 'O'),
('공유진', '1979-04-23', 'M', '010-1111-0027', '경기도 하남시', 'AB'),
('수지은', '1997-03-11', 'F', '010-1111-0028', '서울시 서대문구', 'A'),
('이병헌', '1970-07-12', 'M', '010-1111-0029', '경기도 광명시', 'B'),
('김태희', '1980-03-29', 'F', '010-1111-0030', '서울시 강북구', 'O');

-- Visits (50 rows, roughly last 6 months)
INSERT INTO visits (patient_id, doctor_id, visit_date, visit_type, status, chief_complaint, cost) VALUES
(1,  1,  '2025-11-05', 'outpatient', 'completed', '가슴 통증, 호흡 곤란', 85000),
(2,  5,  '2025-11-08', 'outpatient', 'completed', '자녀 예방접종', 45000),
(3,  7,  '2025-11-10', 'outpatient', 'completed', '허리 통증', 120000),
(4,  9,  '2025-11-12', 'outpatient', 'completed', '여드름 상담', 35000),
(5,  3,  '2025-11-15', 'emergency',  'completed', '복부 통증, 구토', 250000),
(6,  2,  '2025-11-18', 'outpatient', 'cancelled', '기침, 가래', 0),
(7,  11, '2025-11-20', 'outpatient', 'completed', '두통, 어지럼증', 95000),
(8,  15, '2025-11-22', 'outpatient', 'completed', '시력 저하', 75000),
(9,  1,  '2025-11-25', 'outpatient', 'completed', '고혈압 정기검진', 55000),
(10, 13, '2025-11-28', 'outpatient', 'completed', '임신 검진', 65000),
(1,  1,  '2025-12-03', 'outpatient', 'completed', '고혈압 추적검사', 55000),
(11, 3,  '2025-12-05', 'inpatient',  'completed', '담낭 수술', 1500000),
(12, 5,  '2025-12-08', 'outpatient', 'completed', '성장 검진', 40000),
(13, 8,  '2025-12-10', 'outpatient', 'completed', '무릎 통증', 110000),
(14, 17, '2025-12-12', 'outpatient', 'no_show',   '위장 불편', 0),
(15, 11, '2025-12-15', 'outpatient', 'completed', '만성 두통', 95000),
(3,  7,  '2025-12-18', 'outpatient', 'completed', '허리 재진', 80000),
(16, 14, '2025-12-20', 'outpatient', 'completed', '정기 검진', 55000),
(17, 4,  '2025-12-22', 'emergency',  'completed', '교통사고 외상', 350000),
(18, 9,  '2025-12-25', 'outpatient', 'completed', '아토피 상담', 45000),
(19, 11, '2026-01-05', 'outpatient', 'completed', '편두통', 95000),
(20, 2,  '2026-01-08', 'outpatient', 'completed', '감기, 발열', 35000),
(1,  1,  '2026-01-10', 'outpatient', 'completed', '혈압 추적', 55000),
(21, 3,  '2026-01-12', 'inpatient',  'completed', '탈장 수술', 1200000),
(22, 6,  '2026-01-15', 'outpatient', 'completed', '신생아 검진', 50000),
(5,  3,  '2026-01-18', 'outpatient', 'completed', '수술 후 추적검사', 65000),
(23, 15, '2026-01-20', 'outpatient', 'completed', '콘택트렌즈 검사', 45000),
(24, 13, '2026-01-22', 'outpatient', 'completed', '산전 검사', 80000),
(25, 17, '2026-01-25', 'outpatient', 'completed', '소화불량', 45000),
(7,  11, '2026-01-28', 'outpatient', 'completed', '어지럼증 재진', 85000),
(26, 8,  '2026-02-01', 'outpatient', 'completed', '발목 염좌', 95000),
(27, 4,  '2026-02-03', 'outpatient', 'completed', '어깨 통증', 110000),
(28, 10, '2026-02-05', 'outpatient', 'completed', '피부 트러블', 40000),
(29, 12, '2026-02-08', 'outpatient', 'completed', '손 저림', 75000),
(30, 16, '2026-02-10', 'outpatient', 'completed', '안압 검사', 65000),
(2,  6,  '2026-02-12', 'outpatient', 'completed', '영유아 건강검진', 50000),
(4,  10, '2026-02-15', 'outpatient', 'cancelled', '여드름 재진', 0),
(8,  15, '2026-02-18', 'outpatient', 'completed', '시력 재검', 75000),
(10, 14, '2026-02-20', 'outpatient', 'completed', '산후 검진', 60000),
(3,  7,  '2026-02-22', 'outpatient', 'completed', '허리 3차 재진', 80000),
(15, 20, '2026-03-01', 'outpatient', 'completed', '긴장성 두통', 55000),
(9,  17, '2026-03-05', 'outpatient', 'completed', '역류성 식도염', 65000),
(11, 3,  '2026-03-08', 'outpatient', 'completed', '수술 후 6개월 검진', 55000),
(20, 2,  '2026-03-10', 'outpatient', 'completed', '천식 관리', 45000),
(13, 7,  '2026-03-12', 'outpatient', 'completed', '무릎 재활', 90000),
(6,  2,  '2026-03-15', 'outpatient', 'completed', '기관지염', 55000),
(19, 11, '2026-03-18', 'outpatient', 'completed', '두통 추적', 85000),
(25, 1,  '2026-03-20', 'outpatient', 'completed', '건강검진', 120000),
(14, 17, '2026-03-22', 'outpatient', 'completed', '위내시경', 150000),
(1,  1,  '2026-04-01', 'outpatient', 'scheduled', '정기검진 예약', NULL);

-- Diagnoses
INSERT INTO diagnoses (visit_id, icd_code, description, severity) VALUES
(1,  'I20.0', '불안정 협심증', 'moderate'),
(1,  'R06.0', '호흡곤란', 'mild'),
(2,  'Z23',   '예방접종', 'mild'),
(3,  'M54.5', '요통', 'moderate'),
(4,  'L70.0', '심상성 여드름', 'mild'),
(5,  'K35.8', '급성 충수염', 'severe'),
(7,  'G43.9', '편두통', 'moderate'),
(8,  'H52.1', '근시', 'mild'),
(9,  'I10',   '본태성 고혈압', 'mild'),
(10, 'Z34.0', '정상 첫 임신 감독', 'mild'),
(11, 'I10',   '본태성 고혈압', 'mild'),
(12, 'K80.2', '담낭결석', 'severe'),
(13, 'Z00.1', '영유아 건강검진', 'mild'),
(14, 'M17.1', '무릎 골관절염', 'moderate'),
(16, 'G43.0', '전조 없는 편두통', 'moderate'),
(17, 'M54.5', '요통', 'mild'),
(18, 'Z01.4', '부인과 정기검진', 'mild'),
(19, 'S06.0', '뇌진탕', 'severe'),
(19, 'S80.0', '무릎 타박상', 'moderate'),
(20, 'L20.8', '아토피 피부염', 'mild');
"""

with engine.begin() as conn:
    conn.execute(text(seed_data))
print("Seed data inserted.")

In [ ]:
# 시드 데이터가 잘 들어갔는지 테이블별 행 수를 한 번에 확인.
# 학습 포인트:
#   - f-string 의 정렬 문법 `{변수:12s}` → 변수를 12칸짜리 문자열로 좌측 정렬해 표가 깔끔해짐.
#   - `.scalar()` → 결과의 첫 행 첫 컬럼 값 하나만 단순 변수로 받기 (COUNT 같은 단일값에 편리).
for t in ["departments", "doctors", "patients", "visits", "diagnoses"]:
    with engine.connect() as conn:
        # 테이블명을 문자열 보간으로 직접 끼워 넣고 있다는 점에 주의 — 일반적으로는
        # SQL 인젝션 위험이 있어 권장되지 않지만, 여기서는 우리가 만든 고정 리스트라 안전.
        cnt = conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
        print(f"{t:12s}: {cnt} rows")

## SELECT 기본 구조

`SELECT <columns> FROM <table> [WHERE ...] [ORDER BY ...] [LIMIT ...]`

아래 셀들을 순서대로 실행하며 각 절의 역할을 확인합니다. `run_query` 헬퍼는 SQL을 pandas DataFrame으로 실행하고 행 수를 찍어줍니다.

In [ ]:
# Helper: execute a SQL string and return DataFrame
# 주의: LIKE '...강%' 처럼 SQL 본문에 `%` 가 들어가면 psycopg2 의 pyformat 파라미터 자리표시자와 충돌해
# `TypeError: ... immutabledict is not a sequence` 가 납니다. text() 로 감싸 SQLAlchemy 가 자동
# 이스케이프하게 하고, engine 대신 connection 객체를 pandas 에 넘겨 빈 파라미터가 드라이버까지
# 전달되지 않도록 합니다. (SQLAlchemy 2.x + psycopg2 + pandas 2.x 조합 호환성)
def run_query(sql: str, title: str = ""):
    if title:
        print(f"\n[{title}]")
    print(f"SQL: {sql.strip()}\n")
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    print(df.to_string(index=False))
    print(f"({len(df)} rows)")
    return df

In [ ]:
# 모든 컬럼(*) 을 5행만 미리보기. SELECT 의 가장 단순한 형태.
# `LIMIT 5` 가 핵심 — 안 붙이면 전체를 끌어와 결과 출력이 길어집니다.
run_query("SELECT * FROM patients LIMIT 5", "환자 테이블 미리보기")

In [ ]:
# Column selection
run_query("""
    SELECT patient_id, name, gender, blood_type
    FROM patients
    LIMIT 10
""", "환자 이름과 혈액형")

In [ ]:
# WHERE clause
run_query("""
    SELECT name, birth_date, gender
    FROM patients
    WHERE gender = 'F'
    ORDER BY birth_date
""", "여성 환자 (생년월일순)")

In [ ]:
# 계산 컬럼 + 비교 연산
# 학습 포인트:
#   - `AGE(birth_date)` → 생년월일과 오늘 사이의 간격(interval) 을 계산하는 PostgreSQL 함수.
#   - `EXTRACT(YEAR FROM …)` → interval 에서 "년" 부분만 정수로 뽑아 나이를 만든다.
#   - `AS age` → 계산 결과 컬럼에 별칭(alias) 을 부여해 결과 표가 읽기 쉬워진다.
#   - WHERE 와 SELECT 양쪽에 같은 식이 들어간 점 주목 — 별칭(`age`) 은 WHERE 에서 못 쓰기 때문.
run_query("""
    SELECT name, birth_date,
           EXTRACT(YEAR FROM AGE(birth_date)) AS age
    FROM patients
    WHERE EXTRACT(YEAR FROM AGE(birth_date)) >= 40
    ORDER BY birth_date
""", "40세 이상 환자")

In [ ]:
# BETWEEN
run_query("""
    SELECT visit_id, patient_id, visit_date, cost
    FROM visits
    WHERE visit_date BETWEEN '2026-01-01' AND '2026-01-31'
    ORDER BY visit_date
""", "2026년 1월 방문 기록")

In [ ]:
# IN
run_query("""
    SELECT name, blood_type
    FROM patients
    WHERE blood_type IN ('A', 'AB')
    ORDER BY name
""", "혈액형이 A 또는 AB인 환자")

In [ ]:
# LIKE 패턴 매칭 — 와일드카드 `%` 는 "0글자 이상 아무거나" 를 의미합니다.
# - '서울시 강%' → "서울시 강" 으로 시작하는 모든 주소(강남구·강동구·강서구·강북구 …)
# - '%구'        → "구" 로 끝나는 주소
# - '%강%'       → "강" 이 들어간 주소 어디든
# 참고: 본문에 `%` 가 들어 있어도 `run_query` 헬퍼가 text() 로 감싸 주므로 정상 동작합니다.
run_query("""
    SELECT name, address
    FROM patients
    WHERE address LIKE '서울시 강%'
""", "서울시 강~구 거주 환자")

In [ ]:
# IS NULL / IS NOT NULL
run_query("""
    SELECT visit_id, patient_id, visit_date, cost
    FROM visits
    WHERE cost IS NULL OR cost = 0
""", "진료비가 없는 방문 (취소/미방문)")

In [ ]:
# JOIN 첫 등장 — 두 테이블을 "공통 컬럼"으로 연결한다.
# 학습 포인트:
#   - `visits v`, `patients p` → 테이블에 짧은 별칭(v, p) 을 주면 컬럼 참조가 간결해진다.
#   - `JOIN patients p ON p.patient_id = v.patient_id` → 매칭 조건. 양쪽에 모두 있는 행만 남김.
#   - 컬럼 이름이 양쪽에 똑같이 있을 때는 `v.cost` 처럼 별칭으로 명확히 구분해야 모호함이 없다.
#   - `ORDER BY v.cost DESC LIMIT 5` 조합이 "Top-5" 패턴의 표준 SQL 형식.
run_query("""
    SELECT v.visit_id, p.name, v.visit_date, v.cost
    FROM visits v
    JOIN patients p ON p.patient_id = v.patient_id
    WHERE v.cost IS NOT NULL AND v.cost > 0
    ORDER BY v.cost DESC
    LIMIT 5
""", "진료비 상위 5건")

In [ ]:
# 별칭(AS) + 계산 컬럼 + CASE 식 (SQL 의 if/else)
# 학습 포인트:
#   - `AS 이름` → 컬럼명을 한국어로 바꿔 결과 표를 읽기 좋게 만든다.
#   - `CASE gender WHEN 'M' THEN '남' WHEN 'F' THEN '여' END` → 값에 따라 다른 결과를 매핑.
#     Python 의 if/elif/else 와 같은 역할.
#   - ORDER BY 에는 별칭(`나이`) 사용이 가능 — WHERE 와 달리 SELECT 가 먼저 평가되기 때문.
run_query("""
    SELECT
        name AS 이름,
        birth_date AS 생년월일,
        EXTRACT(YEAR FROM AGE(birth_date)) AS 나이,
        CASE gender WHEN 'M' THEN '남' WHEN 'F' THEN '여' END AS 성별
    FROM patients
    ORDER BY 나이 DESC
    LIMIT 10
""", "환자 정보 (한국어 별칭)")

## EXPLAIN — 쿼리 실행 계획 읽기

PostgreSQL은 쿼리를 실행하기 전에 **실행 계획(plan)** 을 세웁니다. `EXPLAIN`은 그 계획을 사람이 읽을 수 있게 보여주며, 성능 튜닝의 출발점입니다.

In [ ]:
# EXPLAIN 은 "쿼리를 실제로 실행하기 전에" 옵티마이저가 세운 계획을 보여 줍니다.
# 학습 포인트:
#   - read_sql 대신 .fetchall() 을 쓰는 이유: 실행 계획은 표(DataFrame) 가 아니라
#     "여러 줄의 텍스트" 라서 그냥 한 줄씩 print 하는 게 더 자연스럽기 때문.
#   - `(FORMAT TEXT)` 옵션을 빼면 기본 형식이 그대로 나오고, 대신 JSON/YAML 형식도 가능.
explain_sql = """
EXPLAIN (FORMAT TEXT)
SELECT p.name, v.visit_date, d.name AS doctor_name
FROM visits v
JOIN patients p ON p.patient_id = v.patient_id
JOIN doctors  d ON d.doctor_id = v.doctor_id
WHERE v.visit_date >= '2026-01-01'
ORDER BY v.visit_date DESC
"""

with engine.connect() as conn:
    plan = conn.execute(text(explain_sql)).fetchall()  # 모든 행을 한꺼번에 받기
    print("Query plan:")
    for row in plan:
        # 각 row 는 한 컬럼짜리 튜플 → row[0] 으로 텍스트만 추출
        print(row[0])

### 실행 계획 읽는 법 (요약)

- **Seq Scan** — 테이블 전체 순차 스캔 (데이터가 많으면 느림)
- **Index Scan** — 인덱스를 사용한 빠른 접근
- **Hash Join / Nested Loop / Merge Join** — 조인 전략
- **Sort** — 정렬 작업
- `cost=시작비용..총비용` — 옵티마이저의 비용 추정치
- `rows=...` — 예상 행 수

오늘은 "실행 계획이라는 것이 존재한다" 정도만 기억해도 충분합니다. 본격적인 튜닝은 본 강의 범위를 벗어납니다.

## 실습 과제

아래 문제를 **다음 셀의 `# TODO`** 영역에 작성해 풀어 보세요.

1. `doctors` 테이블에서 **2020년 이후 입사**한 의사 목록을 급여 내림차순으로 조회하세요.
2. `visits` 테이블에서 **응급(`emergency`) 방문** 기록만 찾아 날짜순으로 정렬하세요.
3. 혈액형이 `O`인 **남성 환자**의 이름과 주소를 조회하세요.
4. 진료비가 **10만원 이상**인 방문에서 환자명·진료일·비용을 조회하세요. (`JOIN` 사용)
5. 과제 4번의 쿼리를 `EXPLAIN`으로 확인하세요.

In [ ]:
# TODO 1: doctors hired in or after 2020, ordered by salary DESC
# 여기에 구현하세요.

# TODO 2: emergency visits ordered by visit_date

# TODO 3: male patients with blood_type = 'O'

# TODO 4: visits with cost >= 100000 joined with patients

# TODO 5: EXPLAIN the query from TODO 4

# 여기에 구현하세요.

## 다음 노트북에서는…

이제 기본 `SELECT`는 익혔습니다. 다음 노트북 **`02_sql_aggregation_join.ipynb`** 에서는 `GROUP BY` / `HAVING`, 다양한 `JOIN`, 서브쿼리, CTE, 윈도우 함수를 실습합니다. 이 노트북에서 만든 병원 DB를 계속 사용하므로 Neon 인스턴스를 유지하세요.